# Task 1: Data Cleaning & Preprocessing

## 1. Loading the Dataset

Loading the raw Netflix dataset using Pandas.

In [7]:
import pandas as pd
import numpy as np
from pathlib import Path

dataset_path = Path("../data/Dataset.csv")

df = pd.read_csv(dataset_path)

print("Dataset loaded successfully!")
print("Shape:", df.shape)

display(df.head())

Dataset loaded successfully!
Shape: (8790, 10)


,show_id,type,title,director,country,date_added,release_year,rating,duration,listed_in
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,United States,9/25/2021,2020,PG-13,90 min,Documentaries
1,s3,TV Show,Ganglands,Julien Leclercq,France,9/24/2021,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act..."
2,s6,TV Show,Midnight Mass,Mike Flanagan,United States,9/24/2021,2021,TV-MA,1 Season,"TV Dramas, TV Horror, TV Mysteries"
3,s14,Movie,Confessions of an Invisible Girl,Bruno Garotti,Brazil,9/22/2021,2021,TV-PG,91 min,"Children & Family Movies, Comedies"
4,s8,Movie,Sankofa,Haile Gerima,United States,9/24/2021,1993,TV-MA,125 min,"Dramas, Independent Movies, International Movies"


## 2. Dataset Structure Inspection

Inspecting column names, data types, non-null counts, and memory usage.


In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8790 entries, 0 to 8789
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   show_id       8790 non-null   str  
 1   type          8790 non-null   str  
 2   title         8790 non-null   str  
 3   director      8790 non-null   str  
 4   country       8790 non-null   str  
 5   date_added    8790 non-null   str  
 6   release_year  8790 non-null   int64
 7   rating        8790 non-null   str  
 8   duration      8790 non-null   str  
 9   listed_in     8790 non-null   str  
dtypes: int64(1), str(9)
memory usage: 686.8 KB


## 3. Missing Values, Placeholders & Duplicates Inspection

In this step, we check data completeness and duplication across multiple levels before making any modifications:
- **Actual nulls (`NaN`)** and **blank/whitespace-only strings**.
- **Text placeholders** representing missing values (such as `"Not Given"` and `"Unknown"`).
- **Exact duplicate rows**, **`show_id` uniqueness**, and **repeated `(title, type)` records**.

In [36]:
obj_cols = df.select_dtypes(include=["object", "string"]).columns

profile = pd.DataFrame({
    "nulls": df[obj_cols].isna().sum(),
    "blanks": [int(df[c].astype(str).str.strip().eq("").sum()) for c in obj_cols],
    "unique": [int(df[c].nunique()) for c in obj_cols],
})
display(profile)

for col in obj_cols:
  if col == "show_id":
    continue
  vc = df[col].value_counts()
  print(f"\n=== {col} (unique={df[col].nunique()}) ===")
  print(vc.to_dict() if df[col].nunique() <= 20 else vc.head(5).to_dict())

,nulls,blanks,unique
show_id,0,0,8790
type,0,0,2
title,0,0,8787
director,0,0,4528
country,0,0,86
date_added,0,0,1713
rating,0,0,14
duration,0,0,220
listed_in,0,0,513



=== type (unique=2) ===
{'Movie': 6126, 'TV Show': 2664}

=== title (unique=8787) ===
{'9-Feb': 2, '15-Aug': 2, '22-Jul': 2, 'Dick Johnson Is Dead': 1, 'Ganglands': 1}

=== director (unique=4528) ===
{'Not Given': 2588, 'Rajiv Chilaka': 20, 'Raúl Campos, Jan Suter': 18, 'Alastair Fothergill': 18, 'Suhas Kadav': 16}

=== country (unique=86) ===
{'United States': 3240, 'India': 1057, 'United Kingdom': 638, 'Pakistan': 421, 'Not Given': 287}

=== date_added (unique=1713) ===
{'1/1/2020': 110, '11/1/2019': 91, '3/1/2018': 75, '12/31/2019': 74, '10/1/2018': 71}

=== rating (unique=14) ===
{'TV-MA': 3205, 'TV-14': 2157, 'TV-PG': 861, 'R': 799, 'PG-13': 490, 'TV-Y7': 333, 'TV-Y': 306, 'PG': 287, 'TV-G': 220, 'NR': 79, 'G': 41, 'TV-Y7-FV': 6, 'NC-17': 3, 'UR': 3}

=== duration (unique=220) ===
{'1 Season': 1791, '2 Seasons': 421, '3 Seasons': 198, '90 min': 152, '97 min': 146}

=== listed_in (unique=513) ===
{'Dramas, International Movies': 362, 'Documentaries': 359, 'Stand-Up Comedy': 334, '

### Scanning for Missing-Value Placeholders

In [37]:
missing_tokens = {
    "not given",
    "not available",
    "unknown",
    "missing",
    "none",
    "null",
    "n/a",
    "na",
    "tbd",
    "not specified",
    "not provided"
}

for col in df.select_dtypes(include=["object", "string"]).columns:
    s = df[col].dropna().astype(str).str.strip()
    hits = s[s.str.lower().isin(missing_tokens)]
    if not hits.empty:
        print(f"\n[{col}]")
        print(hits.value_counts())


[title]
title
Unknown    1
Name: count, dtype: int64

[director]
director
Not Given    2588
Name: count, dtype: int64

[country]
country
Not Given    287
Name: count, dtype: int64


In [35]:
norm_df = df.copy()
str_cols = norm_df.select_dtypes(include=["object", "string"]).columns
for col in str_cols:
  norm_df[col] = norm_df[col].astype(str).str.strip().str.lower()

print("Exact duplicate rows (raw):", int(df.duplicated().sum()))
print("Exact duplicate rows (normalized):", int(norm_df.duplicated().sum()))
print("Unique show_id:", int(df["show_id"].nunique()), "out of", len(df))
print(
    "Duplicate (title, type) pairs (raw):",
    int(df.duplicated(subset=["title", "type"]).sum()),
)
print(
    "Duplicate (title, type) pairs (normalized):",
    int(norm_df.duplicated(subset=["title", "type"]).sum()),
)

dup_records = df[norm_df.duplicated(subset=["title", "type"], keep=False)].copy()
dup_records["_norm_title"] = norm_df.loc[dup_records.index, "title"]
dup_records = dup_records.sort_values(["_norm_title", "show_id"]).drop(
    columns="_norm_title"
)

display(
    dup_records[[
        "show_id",
        "type",
        "title",
        "director",
        "country",
        "date_added",
        "release_year",
        "duration",
    ]]
)

display(df[norm_df["title"] == "unknown"])

Exact duplicate rows (raw): 0
Exact duplicate rows (normalized): 0
Unique show_id: 8790 out of 8790
Duplicate (title, type) pairs (raw): 3
Duplicate (title, type) pairs (normalized): 7


,show_id,type,title,director,country,date_added,release_year,duration
2925,s3963,Movie,15-Aug,Swapnaneel Jayakar,India,3/29/2019,2019,124 min
4261,s5967,Movie,15-Aug,Swapnaneel Jayakar,India,3/29/2019,2019,124 min
3285,s4523,Movie,22-Jul,Paul Greengrass,Norway,10/10/2018,2018,144 min
4260,s5966,Movie,22-Jul,Paul Greengrass,Norway,10/10/2018,2018,144 min
393,s3997,TV Show,9-Feb,Not Given,Pakistan,3/20/2019,2018,1 Season
537,s5965,TV Show,9-Feb,Not Given,Pakistan,3/20/2019,2018,1 Season
2590,s3372,Movie,Consequences,Ozan Açıktan,Turkey,10/25/2019,2014,106 min
4718,s6530,Movie,Consequences,Ozan Açıktan,Turkey,10/25/2019,2014,106 min
415,s304,Movie,Esperando la carroza,Alejandro Doria,Argentina,8/5/2021,1985,95 min
4856,s6706,Movie,Esperando La Carroza,Alejandro Doria,Argentina,7/15/2018,1985,95 min


,show_id,type,title,director,country,date_added,release_year,rating,duration,listed_in
1336,s1468,Movie,Unknown,Jaume Collet-Serra,United Kingdom,1/1/2021,2011,PG-13,113 min,"Action & Adventure, Thrillers"


### Key Conclusions from Step 2

- **`"Unknown"` in `title` (`s1468`) is a valid movie title, not missing data:** Directed by Jaume Collet-Serra (2011) with complete metadata, leaving `"Not Given"` (in `director` and `country`) as the only true missing-value placeholder.
- **Date-like titles (`15-Aug`, `22-Jul`, `9-Feb`) are real titles, not column misalignment:** They reflect actual release titles (*15 August*, *22 July*, *9 February*) that were auto-formatted as short dates when the CSV was previously saved in spreadsheet software.
- **Text normalization uncovered 7 duplicate `(title, type)` pairs (14 rows) of two distinct kinds:** Four pairs share identical `date_added` values (including `Consequences`, hidden by a trailing non-breaking space `\xa0`), while three pairs (`Esperando la carroza`, `Love in a Puff`, `Sin senos sí hay paraíso`) represent catalog re-additions with different `date_added` dates (2018/2019 vs. 2021).